# Clasificación de Tomates en Healthy y Rejected (Exploración Poda)

In [1]:
import tensorflow as tf
import os

## 1. Modelo Base

Modelo basado en el paper "MiniTomatoNet: a lightweight CNN for tomato leaf disease
recognition on heterogeneous FPGA‑SoC"

In [2]:
#Cargar el modelo mintomatonet_v2.keras
modelo = tf.keras.models.load_model("mintomatonet.keras")
modelo.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 120, 120, 3)    │           246 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 120, 120, 18)   │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 30, 30, 18)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 26, 26, 18)     │           468 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 26, 26, 36)     │           684 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 36)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2              │ (None, 3, 3, 36)       │         1,332 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 3, 3, 120)      │         4,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1080)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │       129,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,210 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            22 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 414,584 (1.58 MB)

 Trainable params: 138,194 (539.82 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 276,390 (1.05 MB)

## 2. Carga el dataset

El dataset original: https://dx.doi.org/10.21227/2194-m629 fue reducido para efectos del taller, pero en la realidad debe usarse el dataset completo y sometido a un proceso de balanceo de clases

In [9]:
import tensorflow as tf
from tensorflow import keras
from google.colab import drive
import os

# 1. Montar Google Drive
drive.mount('/content/drive')

# Ruta del dataset en Google Drive
dataset_path = '/content/drive/MyDrive/cursos/Taller_Deeplearning_Villavicencio/dataset_resized_small'

IMG_SIZE = 128
BATCH_SIZE = 8
SEED = 123

if os.path.exists(dataset_path):
    # 2. Carga inicial de todo el dataset (reconoce subcarpetas como clases y crea el vector de etiquetas)
    full_ds = keras.utils.image_dataset_from_directory(
        dataset_path,
        shuffle=True,
        seed=SEED,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE
    )

    class_names = full_ds.class_names
    print("\nClases detectadas:", class_names)

    num_batches = len(full_ds)
    print(f"Total de lotes (batches) cargados: {num_batches}")

    # 3. Calcular los tamaños para la partición: 80% / 10% / 10%
    train_size = int(0.8 * num_batches)
    val_size = int(0.1 * num_batches)
    test_size = num_batches - train_size - val_size

    # Fraccionar usando la API tf.data
    train_ds = full_ds.take(train_size)
    remaining_ds = full_ds.skip(train_size)
    val_ds = remaining_ds.take(val_size)
    test_ds = remaining_ds.skip(val_size)

    print(f"Batches para entrenamiento (80%): {len(train_ds)} (~{len(train_ds) * BATCH_SIZE} imágenes)")
    print(f"Batches para validación (10%): {len(val_ds)} (~{len(val_ds) * BATCH_SIZE} imágenes)")
    print(f"Batches para prueba (10%): {len(test_ds)} (~{len(test_ds) * BATCH_SIZE} imágenes)")

    # Optimización de rendimiento con prefetch
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
    test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)
else:
    print(f"La ruta '{dataset_path}' no es accesible. Por favor verifica que la carpeta exista en tu Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 200 files belonging to 2 classes.

Clases detectadas: ['Healthy', 'Rejected']
Total de lotes (batches) cargados: 25
Batches para entrenamiento (80%): 20 (~160 imágenes)
Batches para validación (10%): 2 (~16 imágenes)
Batches para prueba (10%): 3 (~24 imágenes)


## 3. Funciones auxiliares

### Crear el modelo podado

In [4]:
from tensorflow.keras.layers import (
    Input, Rescaling, SeparableConv2D, DepthwiseConv2D, Conv2D, MaxPooling2D,
    Dense, Flatten
)
from tensorflow.keras.models import Model


def create_model_from_info( info_layers ):
    inputs = Input(shape=(128,128,3))

    # Bloque 1
    x = DepthwiseConv2D(kernel_size=(9,9), padding='valid')(inputs)
    x = Conv2D(filters=info_layers[0]["filters"],
               kernel_size=(1,1),
               activation=info_layers[0]["config"]["activation"],
               padding=info_layers[0]["config"]["padding"])(x)
    x = MaxPooling2D(pool_size=(4,4))(x)

    # Bloque 2
    x = DepthwiseConv2D(kernel_size=(5,5), padding='valid')(x)
    x = Conv2D(filters=info_layers[1]["filters"],
               kernel_size=(1,1),
               activation=info_layers[1]["config"]["activation"],
               padding=info_layers[1]["config"]["padding"])(x)
    x = MaxPooling2D(pool_size=(3,3))(x)

    # Bloque 3
    x = DepthwiseConv2D(kernel_size=(6,6), padding='valid')(x)
    x = Conv2D(filters=info_layers[2]["filters"],
               kernel_size=(1,1),
               activation=info_layers[2]["config"]["activation"],
               padding=info_layers[2]["config"]["padding"])(x)

    x = Flatten()(x)
    x = Dense(units=info_layers[3]["units"],
            activation=info_layers[3]["config"]["activation"],
            use_bias=info_layers[3]["config"]["use_bias"])(x)

    x = Dense(units=info_layers[4]["units"],
                activation=info_layers[4]["config"]["activation"],
                use_bias=info_layers[4]["config"]["use_bias"])(x)

    outputs = Dense(2, activation='softmax')(x)

    return Model(inputs, outputs)


def get_model_info(model):
    info_layers = []
    for layer in model.layers:
        if isinstance(layer, Conv2D):
            info_layers.append({
                "filters": layer.filters,
                "config": layer.get_config(),
                "weights": layer.get_weights(),
            })
        elif isinstance(layer, Dense):
            info_layers.append({
                "units": layer.units,
                "config": layer.get_config(),
                "weights": layer.get_weights(),
            })
    return info_layers


def build_pruned_model(info_layers):
    model = create_model_from_info(info_layers)
    info_idx = 0
    for layer in model.layers:
        if isinstance(layer, (Conv2D, Dense)):
            if (layer.get_weights()[0].shape == info_layers[info_idx]["weights"][0].shape):
                layer.set_weights(info_layers[info_idx]["weights"])
            info_idx += 1
    return model

### Convierte modelo a TFLite

In [15]:
import tensorflow as tf

def representative_data_gen():
    for images, _ in train_ds.take(100):   # usa 100 batches pequeños
        yield [images]

def convert_to_tflite_int8(model, output_file):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_data_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    tflite_model = converter.convert()
    with open(output_file, "wb") as f:
        f.write(tflite_model)

### Exporta los modelos TFLite a C++

In [6]:
# Script en Python puro para emular el comando xxd y exportar el modelo a C
# Comando original: xxd -i mnist_model_quant.tflite > mnist_model_quant.h

def export_tflite_to_c(input_filename, output_filename):
    with open(input_filename, "rb") as f:
        bytes_data = f.read()
    var_name = input_filename.replace(".", "_")
    with open(output_filename, "w") as f:
        f.write(f"// Arreglo de bytes del modelo generado por Python\n")
        f.write(f"const unsigned char {var_name}[] = {{\n    ")
        for i, byte in enumerate(bytes_data):
            f.write(f"0x{byte:02x}")
            if i < len(bytes_data) - 1:
                f.write(", ")
            if (i + 1) % 12 == 0:
                f.write("\n    ")
        f.write("\n};\n\n")
        f.write(f"const unsigned int {var_name}_len = {len(bytes_data)};\n")

    print(f"¡Archivo '{output_filename}' creado con éxito en Python!")

## 4. Poda HRank (%=0.5)

HRank es un método de filter pruning (poda de filtros completos) basado en el rango de los feature maps.
Cada filtro produce un conjunto de feature maps. Si esos feature maps tienen bajo rango, contienen poca información. Esos filtros se pueden eliminar completamente sin afectar mucho la precisión.
HRank solo aplica a capas convolucionales.

In [7]:
import numpy as np

def get_feature_maps(model, layer, data):
    submodel = tf.keras.Model(inputs=model.input, outputs=layer.output)
    feature_maps = submodel.predict(data)
    return feature_maps  # shape: (N, H, W, filters)


def compute_rank_per_filter(feature_maps):
    N, H, W, C = feature_maps.shape
    ranks = []
    for c in range(C):
        fm = feature_maps[..., c]  # (N, H, W)
        rank_sum = 0
        for i in range(N):
            u, s, v = np.linalg.svd(fm[i], full_matrices=False)
            rank_sum += np.sum(s > 1e-5)
        ranks.append(rank_sum / N)
    return np.array(ranks)


def prune_conv_layer(layer, ranks, prune_ratio=0.3):
    num_filters = layer.filters
    num_prune = int(num_filters * prune_ratio)

    idx_prune = np.argsort(ranks)[:num_prune]
    idx_keep = np.setdiff1d(np.arange(num_filters), idx_prune)

    W, b = layer.get_weights()
    W_new = W[..., idx_keep]
    b_new = b[idx_keep]

    return {
        "filters": len(idx_keep),
        "weights": [W_new, b_new],
        "config": layer.get_config()
    }


def aplicar_hrank(model, data_batch, prune_ratio=0.3):
    info_layers = []
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Conv2D):
            # Obtener feature maps
            fm = get_feature_maps(model, layer, data_batch)
            # Calcular rango por filtro
            ranks = compute_rank_per_filter(fm)
            # Podar capa
            info_layers.append( prune_conv_layer(layer, ranks, prune_ratio) )
    return info_layers

In [10]:
info_layers = aplicar_hrank(modelo, test_ds, prune_ratio=0.5)
#Ajusta la información de las capas para reconstruir el modelo podado
info_layers_hrank0_5 = get_model_info(modelo)
info_layers_hrank0_5[:len(info_layers)] = info_layers
model_pruned_hrank0_5 = build_pruned_model(info_layers_hrank0_5)
model_pruned_hrank0_5.summary()

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step  


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 120, 120, 3)    │           246 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 120, 120, 9)    │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 30, 30, 9)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 26, 26, 9)      │           234 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 26, 26, 18)     │           180 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 18)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2              │ (None, 3, 3, 18)       │           666 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 3, 3, 60)       │         1,140 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 540)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        64,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,210 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            22 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,654 (268.18 KB)

 Trainable params: 68,654 (268.18 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Re-entrena el modelo podado

In [ ]:
import matplotlib.pyplot as plt

model_pruned_hrank0_5.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
# Callbacks para evitar overfitting
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(filepath='mejor_modelo.keras',monitor='val_loss',save_best_only=True)
]
# Entrenamiento
historial = model_pruned_hrank0_5.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)

#Graficar la pérdida y la precisión
plt.figure(figsize=(12,5))

# Loss
plt.subplot(1,2,1)
plt.plot(historial.history['loss'], label='Train Loss')
plt.plot(historial.history['val_loss'], label='Val Loss')
plt.title('Loss vs Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy
plt.subplot(1,2,2)
plt.plot(historial.history['accuracy'], label='Train Accuracy')
plt.plot(historial.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy vs Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

#Guarda el modelo entrenado
model_pruned_hrank0_5.save("mintomatonet_pruned_hrank0_5_trained.keras")

#### Exporta el modelo a TFLite

In [ ]:
convert_to_tflite_int8(model_pruned_hrank0_5, "mintomatonet_pruned_hrank0_5_int8.tflite")
export_tflite_to_c( input_filename = "mintomatonet_pruned_hrank0_5_int8.tflite", output_filename = "mintomatonet_pruned_hrank0_5_int8.h" )